# 1. Generating Text with a Pre-trained LLM

---

Packages used in this notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_llm_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_llm_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_llm_from_scratch version: 0.1.0
torch version: 2.14.0
tokenizers version: 0.23.2


## 1.1. Preparing input text for the LLM
We begin by downloading the tokenizer files for the Qwen3 base LLM.

In [2]:
from reasoning_llm_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="../qwen3")

Now, we load the tokenizer settings from the tokenizer file into the `Qwen3Tokenizer`.

In [3]:
from pathlib import Path
from reasoning_llm_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("../qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

Since we haven't loaded the LLM itself yet, we will do a simpler round-trip: we encode the text into token IDs and then decode it back into its string representation.

In [4]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)

In [5]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [6]:
text = tokenizer.decode(input_token_ids_list)
print(text)

Explain large language models.


## 1.2. Loading the pre-trained model

In [7]:
import torch
from reasoning_llm_from_scratch.generate_text_pretrained_llm import get_device

device = get_device()

Using Apple Silicon GPU (MPS)


Then, we download the file containing the pre-trained model weights, which is approximately 1.5 GB in size.

In [8]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="../qwen3")

✓ ../qwen3/qwen3-0.6B-base.pth already up-to-date


In [9]:
from reasoning_llm_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("../qwen3") / "qwen3-0.6B-base.pth"

model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

## 1.3. Coding a minimal text generation function

The `generate_text_basic_stream` function implements a sequential text generation process.

In [10]:
from reasoning_llm_from_scratch.generate_text_pretrained_llm import generate_text_basic_stream

In [11]:
prompt = "Explain large language models in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)
max_new_tokens = 100


for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True  # Deactivates buffering so tokens are printed live
        )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.<|endoftext|>Human language is a complex and dynamic system that has evolved over millions of years to enable effective communication and social interaction. It is composed of a vast array of symbols, including letters, numbers, and words, which are used to convey meaning and express thoughts and ideas. The structure of human language

In [12]:
print(tokenizer.encode("<|endoftext|>"))

[151643]


In [13]:
print(tokenizer.eos_token_id)

151643


In [14]:
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id  # Use EOS token
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Let's implement a simple benchmarking function to track the computational performance.

In [15]:
from reasoning_llm_from_scratch.generate_text_pretrained_llm import generate_stats

In [16]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Time: 0.92 sec
44 tokens/sec
MPS current allocated (tensors): 1.45 GB
MPS driver allocated (incl. cache): 2.17 GB


## 1.4. Faster inference via KV caching

Below is a modified version of the `generate_text_basic_stream` function that uses a KV cache.

In [17]:
from reasoning_llm_from_scratch.generate_text_pretrained_llm import generate_text_basic_stream_cache

In [18]:
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # Collect generated tokens

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, start_time, end_time)

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing essays.

Time: 0.61 sec
57 tokens/sec
MPS current allocated (tensors): 1.45 GB
MPS driver allocated (incl. cache): 2.17 GB


## 1.5. Faster inference via PyTorch model compilation

Another technique to speed up the model inference (text generation) by a lot is using torch.compile.

In [19]:
major, minor = map(int, torch.__version__.split(".")[:2])
if (major, minor) >= (2, 8):
    # This avoids retriggering model recompilations 
    # in PyTorch 2.8 and newer
    # if the model contains code like self.pos = self.pos + 1
    torch._dynamo.config.allow_unspec_int_on_nn_module = True

model_compiled = torch.compile(model)

# If you have issues with torch.compile on "mps" devices and get an InductorError,
# make sure you are using PyTorch 2.9 or newer

First, let's start with the non-cached version.

In [20]:
for i in range(3):

    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()
    

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, start_time, end_time)

    print(f"\n{30*'-'}\n")

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Warm-up run


Time: 5.08 sec
8 tokens/sec
MPS current allocated (tensors): 1.45 GB
MPS driver allocated (incl. cache): 2.17 GB

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 1:


Time: 0.65 sec
62 tokens/sec
MPS current allocated (tensors): 1.45 GB
MPS driver allocated (incl. cache): 2.17 GB

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering

Let's now see how well the KV cache version does.

In [21]:
for i in range(3):
    
    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # Collect generated tokens
    
    end_time = time.time()

    if i == 0:
        print("\n\nWarm-up run")
    else:
        print(f"\n\nTimed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(
        output_token_ids_tensor, start_time, end_time
    )

    print(f"\n{30*'-'}\n")

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Warm-up run


Time: 8.73 sec
4 tokens/sec
MPS current allocated (tensors): 1.45 GB
MPS driver allocated (incl. cache): 2.15 GB

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering questions to writing articles, and even creating creative content.

Timed run 1:


Time: 0.60 sec
68 tokens/sec
MPS current allocated (tensors): 1.45 GB
MPS driver allocated (incl. cache): 2.17 GB

------------------------------

 Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform a wide range of tasks, from answering